In [0]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import current_timestamp, to_utc_timestamp
from pyspark.sql.functions import when, col

In [0]:
dbutils.widgets.text("job_id", "")
job_id = dbutils.widgets.get("job_id")

dbutils.widgets.text("source_file_path", "")
source_file_path = dbutils.widgets.get("source_file_path")

In [0]:
watermark= spark.sql(f"""
SELECT watermark_column 
FROM pt.dev_metadata_schema.tbl_parameters 
where job_id='{job_id}' and source_file_path = '{source_file_path}'
""")
watermark=watermark.collect()[0][0]

In [0]:
print(watermark)

In [0]:

if source_file_path=='dim_tables/tier_details':
    tier_details_df = spark.sql(f"""
        SELECT 
            tier_id,
            tier_name,
            to_timestamp(feed_date) as feed_date
        FROM pt.dev_silver.dim_tier_details
        WHERE to_timestamp(feed_date) > to_timestamp('{watermark}')
        """)
    if tier_details_df.isEmpty():
        dbutils.notebook.exit("no_data")
    else:
        tier_details_df.createOrReplaceTempView("src_dim_tier_details")

elif source_file_path=='dim_tables/products':
    products_df = spark.sql(f"""
        SELECT 
            product_id,
            product_name,
            units,
            stakeholder,
            to_timestamp(feed_date) as feed_date
        FROM pt.dev_silver.dim_products
        WHERE to_timestamp(feed_date) > to_timestamp('{watermark}')
        """)
    if products_df.isEmpty():
        dbutils.notebook.exit("no_data")
    else:
        products_df.createOrReplaceTempView("src_dim_products")

In [0]:
if source_file_path=='dim_tables/tier_details':
        
    spark.sql("""MERGE INTO pt.dev_gold.dim_tier_details AS tgt
    USING src_dim_tier_details AS src

    ON tgt.tier_id = src.tier_id 
    AND tgt.is_active = true

    -- If record exists AND changed → expire old
    WHEN MATCHED AND (
        tgt.tier_name <> src.tier_name OR
        tgt.feed_date <> src.feed_date
    )
    THEN UPDATE SET
        tgt.is_active = false,
        tgt.end_date = current_timestamp()

    --  If new record → insert
    WHEN NOT MATCHED
    THEN INSERT (
        
    tier_id,
    tier_name,
    feed_date,
        is_active,
        start_date,
        end_date
    )
    VALUES (
        
        src.tier_id,
        src.tier_name,
        src.feed_date,
        true,
        current_timestamp(),
        NULL
    )
    """)


elif source_file_path=='dim_tables/products':
    spark.sql("""MERGE INTO pt.dev_gold.dim_products AS tgt
    USING src_dim_products AS src

    ON tgt.product_id = src.product_id 
    AND tgt.is_active = true

    -- 🔁 If record exists AND changed → expire old
    WHEN MATCHED AND (
        tgt.product_name <> src.product_name OR
        tgt.units <> src.units OR
        tgt.stakeholder <> src.stakeholder or
        tgt.feed_date <> src.feed_date
    )
    THEN UPDATE SET
        tgt.is_active = false,
        tgt.end_date = current_timestamp()

    -- 🆕 If new record → insert
    WHEN NOT MATCHED
    THEN INSERT (
       
        product_id,
        product_name,
        units,
        stakeholder,
        feed_date,
        is_active,
        start_date,
        end_date
    )
    VALUES (
        
        src.product_id,
        src.product_name,
        src.units,
        src.stakeholder,
        src.feed_date,
        true,
        current_timestamp(),
        NULL
    )
""")

In [0]:
if source_file_path=='dim_tables/tier_details':
    spark.sql("""INSERT INTO pt.dev_gold.dim_tier_details
    (
        tier_id,
        tier_name,
        feed_date,
        is_active,
        start_date,
        end_date
    )
    SELECT 

        src.tier_id,
        src.tier_name,
        src.feed_date,
        true as is_active,
        current_timestamp() as start_date,
        NULL as end_date
    FROM src_dim_tier_details src
    JOIN pt.dev_gold.dim_tier_details tgt
    ON src.tier_id = tgt.tier_id
    WHERE tgt.is_active = false
    """)

elif source_file_path=='dim_tables/products':
    spark.sql("""INSERT INTO pt.dev_gold.dim_products
    (
        product_id,
        product_name,
        units,
        stakeholder,
        feed_date,
        is_active,
        start_date,
        end_date
    )
    SELECT 

        src.product_id,
        src.product_name,
        src.units,
        src.stakeholder,
        src.feed_date,
        true as is_active,
        current_timestamp() as start_date,
        NULL as end_date
    FROM src_dim_products src
    JOIN pt.dev_gold.dim_products tgt
    ON src.product_id = tgt.product_id
    WHERE tgt.is_active = false
    """)

In [0]:

spark.sql(f"""
UPDATE pt.dev_metadata_schema.tbl_parameters
SET watermark_column = current_timestamp
WHERE job_id='{job_id}' and source_file_path  ='{source_file_path}'
""")

In [0]:
dbutils.notebook.exit("loaded")